In [23]:
import open3d as o3d
import numpy as np
import trimesh
import pymeshlab as ml
from meshlib import mrmeshpy
from tqdm import tqdm

In [24]:
dataset = o3d.data.OfficePointClouds()
pcds = []
for pcd_path in dataset.paths:
    pcds.append(o3d.io.read_point_cloud(pcd_path))

selected_pcd = pcds[13]
# selected_pcd = o3d.io.read_point_cloud("output.pcd")
# selected_pcd = selected_pcd.voxel_down_sample(voxel_size=0.01)
selected_pcd, _ = selected_pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
selected_pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(
    radius=0.1, max_nn=50))  # increase radius
selected_pcd.orient_normals_consistent_tangent_plane(100)
o3d.visualization.draw_geometries([selected_pcd], mesh_show_back_face=True)


In [25]:
distances = selected_pcd.compute_nearest_neighbor_distance()
avg_dist = sum(distances) / len(distances)
radii = [avg_dist * x for x in [0.5, 1.0, 2.0]]

mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
selected_pcd, o3d.utility.DoubleVector(radii))

# pcd_tree = o3d.geometry.KDTreeFlann(selected_pcd)
# mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(selected_pcd, depth=10)[0]
mesh.remove_duplicated_vertices()
mesh.remove_degenerate_triangles()
mesh.remove_unreferenced_vertices()
mesh.remove_non_manifold_edges()

TriangleMesh with 261674 points and 484302 triangles.

In [26]:
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)

[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The handle is invalid. 


In [27]:
o3d.io.write_triangle_mesh("output_mesh.off", mesh)

[Open3D WARNING] Write OFF cannot include triangle normals.


True

In [28]:
import ransac_smoothen_off
input_file = "output_mesh.off"  # Replace with your .OFF file path
output_file = "smoothed_ransac_4.off"
output_dir = "./"  # Directory for comparison images

# Parameters to adjust
min_distance = 0.002  # Minimum distance threshold for high-curvature areas
max_distance = 0.01  # Maximum distance threshold for flat areas
feature_sensitivity = 2.0  # Higher values preserve more features

# Process the mesh
smoothed_mesh = ransac_smoothen_off.smooth_off_file(
    input_file,
    output_file,
    min_distance=min_distance,
    max_distance=max_distance,
    feature_sensitivity=feature_sensitivity,
    visualize=True
)

Loaded mesh with 261590 vertices and 484302 faces
Computing feature importance...
Segmenting planes...
Plane 1 extracted with 51704 points
Plane 2 extracted with 45723 points
Plane 3 extracted with 20954 points
Plane 4 extracted with 14665 points
Plane 5 extracted with 8392 points
Plane 6 extracted with 8050 points
Plane 7 extracted with 6415 points
Plane 8 extracted with 6234 points
Plane 9 extracted with 7235 points
Plane 10 extracted with 7544 points
Plane 11 extracted with 6183 points
Plane 12 extracted with 4250 points
Plane 13 extracted with 5853 points
Plane 14 extracted with 5381 points
Plane 15 extracted with 4113 points
Plane 16 extracted with 3142 points
Plane 17 extracted with 3292 points
Plane 18 extracted with 2850 points
Adding 43446 non-planar points
Found 18 planes
Visualizing segmentation...
Projecting points...
Creating smoothed mesh with texture transfer...
Saving to smoothed_ransac_4.off...


In [29]:
import o3d_off_to_stl
o3d_off_to_stl.convert_off_to_stl("smoothed_ransac_4.off", "output.stl")

Loaded mesh: smoothed_ransac_4.off | Vertices: 261590, Triangles: 484302
Computing vertex normals...
Writing STL to: output.stl
Conversion complete.


In [30]:
mesh = o3d.io.read_triangle_mesh("output.stl")
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)

In [31]:
import visualize_and_compare_off_files
input_file = "smoothed_ransac_4.off" 
visualize_and_compare_off_files.visualize_off_file(input_file)

Mesh loaded: smoothed_ransac_4.off
Vertices: 261590
Triangles: 484302
Has vertex colors: False
Computed vertex normals for visualization.
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The handle is invalid. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The handle is invalid. 


True